# Capsule Networks (CapsNet): Architecture Explained

**Purpose:** Explain *what* a Capsule Network is, *why* its architecture was proposed, and *how* its key layers are implemented.


### Presentation map
| Presenter | Sections | Main responsibility |
|---|---|---|
| **Student 1** | 1–4 | Motivation, Why capsuleNet, Architecture, squash|
| **Student 2** | 5–7 | Convolution, PrimaryCaps, DigitCaps, |
| **Student 3** | 8–11 | Dynamic routing, complete architecture, limitations, applications, recap |






![net](https://files.catbox.moe/0vkq71.svg)

---
# PART I

## 1. What is a Capsule Network?

A **Capsule Network (CapsNet)** is a neural network architecture (were originally developed for image recognition tasks).
**It is called a capsule because it groups multiple neurons into a single vector-output unit that represents a visual entity and its properties.**

that represents features using **vectors** rather than relying only on individual scalar activations. A *capsule* is a group of neurons whose combined output forms a vector.
![Capsule Network](https://files.catbox.moe/46sj4o.png)

**A vector has two key properties:**
- The **length** of the vector is designed to reflect how strongly the capsule detects an entity.

- The **orientation** can encode learned properties of that entity.




## 2. Why CapsNet?

A conventional **Convolutional Neural Network (CNN)** learns image features through convolutional filters. Many CNN architectures also use **pooling**, which reduces spatial resolution. Pooling can be useful for robustness and efficiency, but it may discard some detailed location information.

CapsNet was proposed to model relationships between parts and wholes more explicitly.

 Its lower-level capsules produce *predictions* about higher-level capsules; the network then uses **agreement** between these predictions to decide how strongly they should contribute.
 ![Network](https://files.catbox.moe/vpybfr.svg)


**Scientific caution:** It is incorrect to say “CNNs cannot learn spatial relationships” or “CapsNets always beat CNNs.” Modern CNNs can capture spatial structure; relative performance depends on the task, implementation, and data. The survey itself discusses cases where CNNs outperform CapsNets (Sections 7–8).



## 3. CapsNet Architecture Overview
The original CapsNet architecture consists of a:

**- Convolutional layer**

**- PrimaryCaps layer**

**- DigitCaps layer**.
![Architecture](https://files.catbox.moe/8fr5qz.svg)
We will explain how each layer works in the following sections.



## 4. Squashing function: controlling vector length

Why not pass capsule vectors directly to the next layer? Their raw lengths may vary widely. The **squash** function keeps the direction of a nonzero vector while compressing its length to the interval **[0, 1]**:
تنزيل.svg


### Squash Function Example

The squash function is defined as:

$$
\operatorname{squash}(\mathbf{s})=
\frac{\|\mathbf{s}\|^2}{1+\|\mathbf{s}\|^2}
\frac{\mathbf{s}}{\|\mathbf{s}\|}
$$

The formula has two parts:

1. $\frac{\mathbf{s}}{\|\mathbf{s}\|}$ preserves the original **direction**.
2. $\frac{\|\mathbf{s}\|^2}{1+\|\mathbf{s}\|^2}$ determines the **new length**.

**Example:** Suppose the input vector is $\mathbf{s}=(3,4)$.

**Step 1: Calculate the original vector length.**

$$
\|\mathbf{s}\|=\sqrt{3^2+4^2}=5
$$

**Step 2: Calculate the direction.**

$$
\frac{\mathbf{s}}{\|\mathbf{s}\|}=\frac{(3,4)}{5}=(0.6,0.8)
$$

**Step 3: Calculate the new length.**

$$
\frac{\|\mathbf{s}\|^2}{1+\|\mathbf{s}\|^2}
=\frac{25}{26}\approx0.962
$$

**Step 4: Calculate the final squashed vector.**

$$
\operatorname{squash}(\mathbf{s})
=0.962(0.6,0.8)
\approx(0.577,0.769)
$$

**Result:**

- Original vector length: **5**
- New vector length: **0.962**
- Direction: **Unchanged**

The squash function reduces the vector length to a value between 0 and 1 while preserving its direction.


---
# PART II



##5. Input Layer — Receive the Image

**Input**:
Our image comes from MNIST, which contains handwritten digits.
The input image is:
$$
28\times28\times1
$$

What do these numbers mean?
- First 28: image height in pixels.
- Second 28: image width in pixels.
- The 1: one color channel (grayscale).

$$
\boxed{28\times28\times1}
$$

This image is passed to Conv1.

##6. Conv1 — Detect Visual Features
**Input**: 28 × 28 × 1 image.

**Process:**

Conv1 is a convolutional layer that uses filters to detect patterns such as edges, curves, and strokes.

* Kernel (9 × 9): A small grid of learned weights that scans the image.

* Stride (1): The kernel moves one pixel at a time.

* 256 filters: Each filter learns to detect useful visual patterns.

Each of the 256 filters produces a 20 × 20 feature map.

**Output:**
$$
\boxed{20\times20\times256}
$$

We now have 256 feature maps containing numerical responses to visual patterns.

![abc](https://files.catbox.moe/09t6yp.png)
![ab2](https://files.catbox.moe/h051gl.gif)

##Step 7 — PrimaryCaps - Group into Capsules

###Step 1 — Receive features from Conv1
**Input:** 20 × 20 × 256

PrimaryCaps applies a second convolution with:
- Kernel: 9 × 9
- Stride: 2


**Intermediate Output:** 6 × 6 × 256


###Step 2 — Organize the 256 channels into capsules

Divide the 256 values into groups of 8:

$$
256\div8=32\text{ capsules}
$$

![abc4](https://files.catbox.moe/xmcepy.png)

### Step 3 — Count all capsules

The output has a `6 × 6` grid, meaning 36 positions.

Each position contains 32 capsules and the sqash equation applied to each capsul

$$
6\times6\times32=1152\text{ capsules}
$$

**Final output:**

$$
\boxed{1152\times8}
$$

This means **1,152 capsules, each containing 8 numbers**.

![abc5](https://files.catbox.moe/5sgi8w.png)

###PrimaryCaps Summary

* Input: 20 × 20 × 256

* Convolution → 6 × 6 × 256

* Group into capsules → 1152 × 8

* Apply Squash → 1152 × 8

* **Output**: 1,152 capsules, each represented by an 8D vector.



![abc3](https://files.catbox.moe/tlsv9n.png)


##8. DigitCaps — Recognizing Complete Digits
Main idea: PrimaryCaps detects and describes parts of the image. through combines these parts to recognize a complete digit (0–9).


Step 1 — Receive Vectors from PrimaryCaps (1152 × 8)

Step 2 — Make Predictions for each digit


There are 10 possible digits (0–9).

Each of the 1,152 capsules makes a prediction for all 10 digits.

A **(Dynamic Routing)** converts each 8D vector into a 16D vector.

Final DigitCaps output:
$$
\boxed{10\times16}
$$

![abc](https://files.catbox.moe/jtljmu.png)

##One-sentence explanation of each layer
- Convolution: detects useful local image features.
- PrimaryCaps: groups local features into 8D vectors for lower-level entities.
- DigitCaps: combines transformed capsule predictions, via dynamic routing, into 16D vectors for whole digit classes.


---
# PART III

## 8. Dynamic Routing by Agreement

**Dynamic routing** is the mechanism that determines how much each lower-level capsule contributes to each higher-level capsule. So, each lower capsule sends its vote to the higher capsule it **agrees** with, and the final prediction is the **longest** output vector.

### The problem
After DigitCaps, every one of the **10 digit capsules** receives **1,152 votes**, which is **11,520 votes** per image. Most of them come from parts that belong to a different digit.

> **Question:** which votes should each digit capsule trust?


**Analogy:** three people estimate where an object is. If two of them point to nearly the same spot, that is stronger evidence than three unrelated guesses. Routing is a mathematical version of this idea, comparing **vectors** instead of spoken opinions.


### Example: routing on a toy example
**3 lower capsules** ($u_1$, $u_2$, $u_3$) vote for **2 higher capsules** (A and B). The real model has 1,152 and 10, but the steps are the same.

**How to read the figure**
- **Line thickness** = coupling coefficient $c_{ij}$, meaning how much $u_i$ contributes to higher capsule $j$.
- **Arrow in a right-hand circle** = that capsule's output vector. A longer arrow means more confidence.
- $u_1$ and $u_2$ vote for A in similar directions (they **agree**). $u_3$'s vote for A points the opposite way (an **outlier**).

![routing](https://files.catbox.moe/1ygezg.png)

**What to notice**
1. **Iteration 1:** every line is equal, so there is no preference yet.
2. **Iteration 2:** the agreeing votes ($u_1$, $u_2$) gain weight toward A.
3. **Iteration 3:** $u_3$ has moved almost entirely to B, and Digit A's output vector is longer.

**Takeaway:** routing uses no labels and no loss. It only checks whether votes *agree*, and it strengthens the connections where they do. The coupling coefficients are recomputed for **every image**, so the connections change from input to input.

### What happens during routing?
1. **Generate votes:** Each PrimaryCaps capsule predicts a 16D vector for each DigitCaps capsule.
2. **Initialize routing logits:** Start with $b_{ij}=0$ for all input–output pairs. These are temporary routing scores, not the learned transformation weights.
3. **Compute coupling coefficients:** Apply **softmax across the 10 output capsules** for each input capsule. Initially, each receives $1/10=0.1$.
4. **Weighted sum:** Combine the votes for each digit capsule according to their coefficients.
5. **Squash:** Convert that sum to an output capsule vector.
6. **Measure agreement:** Take a dot product between each vote and its corresponding output capsule. Increase routing logits where agreement is stronger.
7. **Repeat:** In this notebook, repeat for **3 routing iterations**.


In [ ]:
def dynamic_routing(votes, iterations=3):
    """Route votes [B, N_input, N_output, D_output]."""
    batch_size, n_input, n_output, d_output = votes.shape

    # b_ij: initial routing logits, separate for every image in the batch.
    logits = votes.new_zeros(batch_size, n_input, n_output)

    for step in range(iterations):
        # For each input capsule, distribute its weight over output capsules.
        coupling = torch.softmax(logits, dim=2)
        # coupling: [B, N_input, N_output]

        # Sum weighted 16D votes to create one vector per output capsule.
        weighted_votes = coupling.unsqueeze(-1) * votes
        summed = weighted_votes.sum(dim=1)   # [B, N_output, D_output]
        output = squash(summed)              # [B, N_output, D_output]

        if step < iterations - 1:
            # Agreement: dot product between each vote and output capsule.
            agreement = (votes * output.unsqueeze(1)).sum(dim=-1)
            logits = logits + agreement

    return output

## 9. Limitations and challenges

These are the challenges **reported in the 2023 survey**.

| Challenge | In short |
|---|---|
| **Computational cost** | 1,152 × 10 votes, repeated over several routing iterations, need more compute and memory |
| **Scaling** | MNIST is small and simple. Good results there do **not** prove CapsNet works on complex real-world images |
| **Routing design** | The number of iterations is a choice, and many alternative routing methods have been proposed |
| **Interpretability** | The vectors are meant to encode properties, but giving each component a human meaning is hard |
| **Robustness** | No guaranteed invariance or protection against adversarial attacks, and results are mixed against CNNs |

**Bottom line:** CapsNet offers a distinctive way to represent structure, with trade-offs in computation, design, and practical performance.

## 10. Applications and final summary

| Application | Why researchers explore capsules |
|---|---|
| **Handwritten digit recognition** | Shows vector capsules and routing on a simple task |
| **Medical image analysis** | Relationships between anatomical structures and features |
| **Object recognition and pose estimation** | Parts, arrangements, and viewpoint-related properties |
| **Natural language processing** | Hierarchies among words and phrases |
| **Video analysis** | Structured spatial or temporal patterns |

These are **research directions**, not proof that CapsNet is the best choice in each area.


### Finally
> Capsule Networks extend conventional feature extraction with vector representations and dynamic routing. Their main contribution is a mechanism for modeling **agreement between lower-level parts and higher-level entities**. They are conceptually interesting, but practical performance depends on the problem, and the computational cost is real.



# Appendix A — References and source mapping

**Primary paper**  
Sabour, S., Frosst, N., & Hinton, G. E. (2017). *Dynamic Routing Between Capsules*. Advances in Neural Information Processing Systems (NeurIPS). https://arxiv.org/abs/1710.09829

**Additional survey (user-provided PDF)**  
Haq, M. U., Sethi, M. A. J., & Rehman, A. U. (2023). *Capsule Network with Its Limitation, Modification, and Applications—A Survey*. *Machine Learning and Knowledge Extraction, 5*, 891–921. https://doi.org/10.3390/make5030047


##9.Tensor dimension
A tensor is an array of numbers. Its shape tells us how the numbers are arranged.
Example	Shape	Meaning


| Stage	| Tensor shape (one image)	| Meaning|
|---| --- | --- |
|Input | `1× 28 × 28` | Grayscale image|
|Conv1 |`256 × 20 × 20` | 256 feature maps |
|PrimaryCaps (before grouping) |	`256 × 6 × 6` |	Convolution outputs |
|PrimaryCaps (grouped) |	`6 × 6 × 32 × 8`	| 8D vectors|
|PrimaryCaps (flattened) |	`1152 × 8`	 | 1,152 capsules|
|DigitCaps |	`10 × 16` | 10  desdigit representations|
|Classification |	`10` |	One length per digit|